In [ ]:
!pip install streamlit -q
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
changed 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics

# --- PAGE CONFIGURATION ---
st.set_page_config(page_title="Indian House Price Prediction System", layout="wide")
st.title("🏢 House Price Prediction System")
st.markdown("### **Objective:** Predict house prices based on realistic area, location, rooms, and infrastructure data.")

# --- CACHED DATA, FEATURE ENGINEERING & MODEL TRAINING ---
@st.cache_resource
def load_and_train_system():
    # 1. Data Collection
    url = "https://raw.githubusercontent.com/karlyndiary/Real-Estate-Price-Prediction/main/Bengaluru_House_Data.csv"
    df_raw = pd.read_csv(url)

    # 2. Data Cleaning
    df = df_raw.drop(['area_type', 'society', 'balcony', 'availability'], axis=1).dropna()
    df['BHK'] = df['size'].apply(lambda x: int(x.split(' ')[0]))
    df = df.drop(['size'], axis=1)

    def convert_sqft_to_num(x):
        tokens = str(x).split('-')
        if len(tokens) == 2:
            return (float(tokens[0]) + float(tokens[1])) / 2
        try:
            return float(x)
        except:
            return None

    df['total_sqft'] = df['total_sqft'].apply(convert_sqft_to_num)
    df = df.dropna()

    # --- REAL-WORLD OUTLIER FILTERING ---
    # 1. Remove properties where BHK size makes no sense (less than 300 sqft per BHK)
    df = df[(df['total_sqft'] / df['BHK']) >= 300]

    # 2. Calculate price per sqft for outlier detection
    df['price_per_sqft'] = (df['price'] * 100000) / df['total_sqft']

    # 3. Filter price outliers per location using mean and 1 standard deviation
    def remove_pps_outliers(df):
        df_out = pd.DataFrame()
        for key, subdf in df.groupby('location'):
            m = np.mean(subdf.price_per_sqft)
            st_dev = np.std(subdf.price_per_sqft)
            reduced_df = subdf[(subdf.price_per_sqft > (m - st_dev)) & (subdf.price_per_sqft <= (m + st_dev))]
            df_out = pd.concat([df_out, reduced_df], ignore_index=True)
        return df_out

    df = remove_pps_outliers(df)
    df = df.drop(['price_per_sqft'], axis=1)

    # --- LOGICAL FEATURE ENGINEERING (Instead of pure random noise) ---
    # Tie features to logic: High price-point / big area properties get better amenities
    np.random.seed(42)
    df['property_age'] = np.random.randint(0, 20, size=len(df)) # Restricting to active lifespans

    # Logic: If a house is large or has high price, it's more likely to have premium amenities
    df['has_swimming_pool'] = (df['total_sqft'] > 1800).astype(int)
    df['has_gym'] = (df['BHK'] >= 3).astype(int)
    df['has_power_backup'] = np.random.choice([0, 1], size=len(df), p=[0.1, 0.9]) # Standard feature in apartments
    df['has_security'] = np.random.choice([0, 1], size=len(df), p=[0.1, 0.9])

    # Location grouping logic (Keep locations with more than 20 instances for better pattern matching)
    location_counts = df['location'].value_counts()
    valid_locations = sorted(list(location_counts[location_counts > 20].index))
    df['location'] = df['location'].apply(lambda x: x if x in valid_locations else 'other')

    # One-Hot Encoding
    final_df = pd.get_dummies(df, columns=['location'], drop_first=True, dtype=int)

    # Split features
    X = final_df.drop(['price'], axis=1)
    Y = final_df['price']

    # Train/Test Split
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

    # XGBoost Hyperparameter Tuning for Real Estate (Prevents Overfitting)
    model = XGBRegressor(n_estimators=100, learning_rate=0.08, max_depth=6, random_state=42)
    model.fit(X_train, Y_train)

    # Evaluation Metrics
    predictions = model.predict(X_test)
    r2_score = metrics.r2_score(Y_test, predictions)
    mae_score = metrics.mean_absolute_error(Y_test, predictions)

    model_features = X.columns.tolist()

    return model, valid_locations, model_features, r2_score, mae_score

# Run pipeline
with st.spinner("Processing Dataset & Removing Outliers..."):
    model, locations_list, model_features, r2_accuracy, mae_error = load_and_train_system()

# --- SIDEBAR METRICS DISPLAY ---
st.sidebar.header("📊 Model Evaluation Status")
st.sidebar.metric(label="R-squared (Accuracy)", value=f"{r2_accuracy:.2f}")
st.sidebar.metric(label="Mean Absolute Error (MAE)", value=f"± {mae_error:.2f} Lakhs")

# --- USER FRIENDLY INTERACTIVE FORM ---
st.header("🔑 Enter Property Parameters")

col1, col2, col3 = st.columns(3)

with col1:
    selected_location = st.selectbox("Select Target Location", options=locations_list)
    total_sqft = st.number_input("Total Area (Square Feet)", min_value=300, max_value=10000, value=1200, step=50)

with col2:
    bhk = st.slider("Number of Rooms (BHK)", min_value=1, max_value=8, value=2)
    bath = st.slider("Number of Bathrooms", min_value=1, max_value=6, value=2)

with col3:
    property_age = st.slider("Age of the Property (Years)", min_value=0, max_value=30, value=2)

st.markdown("---")
st.subheader("🏊‍♂️ Select Available Amenities")
amenity_col1, amenity_col2, amenity_col3, amenity_col4 = st.columns(4)

with amenity_col1:
    pool_opt = st.checkbox("Swimming Pool (Recommended for properties > 1800 sqft)")
with amenity_col2:
    gym_opt = st.checkbox("Gym (Recommended for 3+ BHK)")
with amenity_col3:
    backup_opt = st.checkbox("24/7 Power Backup", value=True)
with amenity_col4:
    security_opt = st.checkbox("24/7 Gated Security", value=True)

# --- MODEL INFERENCE PIPELINE ---
if st.button("💰 Calculate Estimated Valuation", type="primary", use_container_width=True):
    input_data = pd.DataFrame(0, index=[0], columns=model_features)

    input_data['total_sqft'] = total_sqft
    input_data['bath'] = bath
    input_data['BHK'] = bhk
    input_data['property_age'] = property_age

    input_data['has_swimming_pool'] = 1 if pool_opt else 0
    input_data['has_gym'] = 1 if gym_opt else 0
    input_data['has_power_backup'] = 1 if backup_opt else 0
    input_data['has_security'] = 1 if security_opt else 0

    target_location_column = f"location_{selected_location}"
    if target_location_column in input_data.columns:
        input_data[target_location_column] = 1

    predicted_val_lakhs = model.predict(input_data)[0]

    if predicted_val_lakhs < 0:
        predicted_val_lakhs = abs(predicted_val_lakhs)

    st.markdown("---")
    st.subheader("🎯 Final AI Model Output")

    if predicted_val_lakhs >= 100:
        crores = predicted_val_lakhs / 100
        st.success(f"### Predicted Price: **₹ {crores:.2f} Crores** ({predicted_val_lakhs:.1f} Lakhs)")
    else:
        st.success(f"### Predicted Price: **₹ {predicted_val_lakhs:.2f} Lakhs**")

Overwriting app.py


In [ ]:
!wget -qO- ipv4.icanhazip.com

34.82.124.156


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋

⠙⠹⠸⠼⠴your url is: https://famous-rings-rhyme.loca.lt
2026-07-16 18:31:08.572 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.82.124.156:8501

